In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import optuna
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = "/workspace/OCT2017"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")

IMG_SIZE   = 224
BATCH_SIZE = 32


DATA_PERCENT = 0.50    
EPOCHS_PER_TRIAL = 25
N_TRIALS = 20


In [ ]:
import os
import random

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)

IMAGES_PER_CLASS = 4000 

def sample_exact_per_class(base_dir, per_class):
    all_paths = []
    all_labels = []
    
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(base_dir, class_name)

        files = [
            os.path.join(class_dir, f)
            for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
        ]

        random.shuffle(files)

        k = min(per_class, len(files))
        selected = files[:k]

        print(f"{class_name}: selected {k} images")

        all_paths.extend(selected)
        all_labels.extend([class_idx] * k)

    return np.array(all_paths), np.array(all_labels)

train_paths, train_labels = sample_exact_per_class(TRAIN_DIR, IMAGES_PER_CLASS)


In [ ]:
def load_val_paths():
    paths = []
    labels = []
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(VAL_DIR, class_name)
        files = [
            os.path.join(class_dir, f)
            for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
        ]
        paths.extend(files)
        labels.extend([class_idx]*len(files))
    return np.array(paths), np.array(labels)

val_paths, val_labels = load_val_paths()
print("Validation images:", len(val_paths))



In [ ]:
def create_clahe(clip_limit, grid_size):
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))


def apply_clahe_np(img, clip_limit, grid_size):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = create_clahe(clip_limit, grid_size)
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    rgb = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return rgb.astype(np.float32) / 255.0


In [ ]:
def make_loader(clip_limit, grid_size):
    def _loader(path, label):
        def _py_func(p):
            img = cv2.imread(p.decode())
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = apply_clahe_np(img, clip_limit, grid_size)
            return img

        img = tf.numpy_function(_py_func, [path], tf.float32)
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        return img, label
    return _loader


In [ ]:
def make_dataset(paths, labels, clip_limit, grid_size, shuffle=False):
    loader = make_loader(clip_limit, grid_size)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths))
    ds = ds.map(loader, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds



In [ ]:
def build_model(lr):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.05)(x)
    x = layers.RandomContrast(0.03)(x)

    # Block 1
    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 2
    x = layers.Conv2D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 3
    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 4
    x = layers.Conv2D(512, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 5
    x = layers.Conv2D(512, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.6)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
def objective(trial):

    lr = trial.suggest_float("lr", 1e-6, 3e-4, log=True)
    clip_limit = trial.suggest_float("clip_limit", 0.8, 3.0)
    grid_size = trial.suggest_categorical("grid_size", [4, 6, 8])

    train_ds = make_dataset(train_paths, train_labels, clip_limit, grid_size, shuffle=True)
    val_ds   = make_dataset(val_paths, val_labels, clip_limit, grid_size)

    model = build_model(lr)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_PER_TRIAL,
        verbose=0
    )

    return history.history["val_accuracy"][-1]


In [ ]:
import time

trial_start_times = {}

def trial_start_callback(study, trial):
    trial_start_times[trial.number] = time.time()

def trial_end_callback(study, trial):
    end_time = time.time()
    start_time = trial_start_times.get(trial.number, end_time)
    duration = end_time - start_time

    print("\n" + "="*70)
    print(f"TRIAL {trial.number} FINISHED")
    print(f"Time Taken: {duration:.2f} seconds")
    print(f"Validation Accuracy: {trial.value}")
    print(f"Parameters: {trial.params}")
    print("="*70 + "\n")


study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    callbacks=[trial_start_callback, trial_end_callback]
)

print("\n==================== OVERALL BEST ====================")
print("Best Params:", study.best_params)
print("Best Value:", study.best_value)
print("======================================================")
